In [1]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna higher

In [2]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import mplfinance as mpf
import pandas_ta as ta
import pygame
import os
import pytz
import json
import re
import random
import pickle

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, Attention, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.callbacks import Callback, ModelCheckpoint
from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque
from tensorflow import keras
from tensorflow.keras import layers, optimizers, models
from scipy.signal import find_peaks

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 25  # 25 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 15  # 15 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [4]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [5]:
 session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [6]:
def fetch_candle_data(number, index_symbol, interval_minutes):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [7]:
def fetch_train_candle_data(days_count, index_symbol, interval_minutes):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [8]:
two_interval_minutes = 2

nf_index_symbol, nf_quantity = get_index_symbol_and_quantity("Nifty")
bnf_index_symbol, bnf_quantity = get_index_symbol_and_quantity("Bank Nifty")

#nf_train_df = fetch_candle_data(100, nf_index_symbol, two_interval_minutes)
#bnf_train_df = fetch_candle_data(100, bnf_index_symbol, two_interval_minutes)

nf_train_df = fetch_train_candle_data(1, nf_index_symbol, two_interval_minutes)
bnf_train_df = fetch_train_candle_data(1, bnf_index_symbol, two_interval_minutes)

print(len(nf_train_df), len(bnf_train_df))

nf_train_df = nf_train_df.drop_duplicates(subset='datetime', keep='first')
bnf_train_df = bnf_train_df.drop_duplicates(subset='datetime', keep='first')

print(len(nf_train_df), len(bnf_train_df))

13160 13160
13160 13160


In [9]:
class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame, base_interval_minutes: int):
        """
        df: DataFrame with columns [datetime, open, high, low, close, volume (optional)]
            'datetime' is a Unix timestamp in seconds.
        base_interval_minutes: The base timeframe in minutes (e.g. 2, 5, 15, etc.).
        """
        self.df = df.copy()
        self.base_interval = base_interval_minutes

        # We no longer need an htf_map since higher timeframe features are removed
        # self.htf_map = { ... }  # Removed

    def preprocess_datetime(self):
        """
        1) Convert Unix timestamp to local time (Asia/Kolkata).
        2) Check for duplicates/missing timestamps.
        3) Sort by time and set 'datetime' as the index.
        """
        ist = timezone('Asia/Kolkata')

        self.df['datetime'] = pd.to_datetime(self.df['datetime'], unit='s')
        self.df['datetime'] = (self.df['datetime']
                               .dt.tz_localize('UTC')
                               .dt.tz_convert(ist)
                               .dt.tz_localize(None))

        # Check for duplicates/missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)
        return self

    def clean_data(self):
        """
        Optionally drop 'volume' column if it contains zeros or NaNs.
        Adjust if you prefer different volume handling.
        """
        if 'volume' in self.df.columns:
            if self.df['volume'].isnull().any() or (self.df['volume'] == 0).any():
                self.df.drop('volume', axis=1, inplace=True, errors='ignore')
        return self

    def add_base_timeframe_features(self):
        """
        Compute a few standard technical indicators on the base timeframe:
        RSI, MACD, Bollinger Bands, ATR, etc.
        """
        self.df.sort_index(inplace=True)

        # RSI on base timeframe close
        self.df['rsi_base'] = ta.rsi(self.df['close'], length=14)

        # MACD
        macd = ta.macd(self.df['close'], fast=12, slow=26, signal=9)
        self.df['macd'] = macd['MACD_12_26_9']
        self.df['macd_h'] = macd['MACDh_12_26_9']
        self.df['macd_s'] = macd['MACDs_12_26_9']

        # Bollinger Bands
        bbands = ta.bbands(self.df['close'], length=20, std=2)
        self.df['bb_lower'] = bbands['BBL_20_2.0']
        self.df['bb_mid'] = bbands['BBM_20_2.0']
        self.df['bb_upper'] = bbands['BBU_20_2.0']

        # ATR (for baseline volatility measure)
        self.df['atr_base'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        )

        return self

    def add_candlestick_patterns(self):
        """
        Detect a few common candlestick patterns.
        We store them as numeric columns (0 = not present, 1 = bullish pattern, -1 = bearish pattern, etc.)
        so that the pipeline remains consistent with numeric-only scaling.
        """
        self.df.sort_index(inplace=True)

        # Example: Doji detection (very simplistic)
        # We check if the open and close are nearly the same
        body_size = (self.df['close'] - self.df['open']).abs()
        candle_range = self.df['high'] - self.df['low']
        self.df['candle_doji'] = np.where(
            (body_size <= 0.1 * candle_range), 1.0, 0.0
        )

        # Example: Engulfing pattern (bullish and bearish)
        # - Bullish Engulfing: today's open < yesterday's close, today's close > yesterday's open
        # - Bearish Engulfing: today's open > yesterday's close, today's close < yesterday's open
        # We shift the open/close by 1 day to compare
        self.df['open_shift1'] = self.df['open'].shift(1)
        self.df['close_shift1'] = self.df['close'].shift(1)

        cond_bull_engulf = (self.df['open'] < self.df['close_shift1']) & (self.df['close'] > self.df['open_shift1'])
        cond_bear_engulf = (self.df['open'] > self.df['close_shift1']) & (self.df['close'] < self.df['open_shift1'])

        self.df['candle_engulf'] = np.where(cond_bull_engulf, 1.0,
                                    np.where(cond_bear_engulf, -1.0, 0.0))

        # Drop helper columns to keep dataset clean
        self.df.drop(['open_shift1', 'close_shift1'], axis=1, inplace=True)

        return self

    def add_swing_points(self, left_bars=3, right_bars=3):
        """
        Detect local swing highs and lows.
        left_bars/right_bars define how many bars on each side must be lower/higher (for highs/lows).
        """
        self.df.sort_index(inplace=True)

        # For each row, check if 'high' is greater than the preceding and following N bars
        rolling_high = self.df['high'].rolling(left_bars+1, center=False).max()
        rolling_low = self.df['low'].rolling(left_bars+1, center=False).min()

        # We'll do a simplistic approach: shift rolling computations accordingly
        # A more robust approach might be to iterate row by row, but we'll keep it vectorized for brevity.

        # Swing High
        # Compare current high to future bars as well, so we do a forward rolling
        forward_high = self.df['high'].shift(-right_bars).rolling(right_bars+1, center=False).max()
        self.df['is_swing_high'] = 0.0
        cond_high = (self.df['high'] == rolling_high) & (self.df['high'] == forward_high)
        self.df.loc[cond_high, 'is_swing_high'] = 1.0

        # Swing Low
        forward_low = self.df['low'].shift(-right_bars).rolling(right_bars+1, center=False).min()
        self.df['is_swing_low'] = 0.0
        cond_low = (self.df['low'] == rolling_low) & (self.df['low'] == forward_low)
        self.df.loc[cond_low, 'is_swing_low'] = 1.0

        return self

    def add_range_breakout_features(self, window=20):
        """
        Add Donchian channels, breakout flags, and a simple measure of range expansion.
        """
        self.df.sort_index(inplace=True)

        # Donchian Channels
        self.df['donchian_high'] = self.df['high'].rolling(window).max()
        self.df['donchian_low'] = self.df['low'].rolling(window).min()
        self.df['donchian_range'] = self.df['donchian_high'] - self.df['donchian_low']

        # Breakout flags
        #  1.0 if close > donchian_high, -1.0 if close < donchian_low, else 0.0
        self.df['donchian_breakout'] = np.where(
            self.df['close'] > self.df['donchian_high'], 1.0,
            np.where(self.df['close'] < self.df['donchian_low'], -1.0, 0.0)
        )

        # Range expansion/compression example
        # Rolling std of (high - low)
        self.df['range_expansion'] = (self.df['high'] - self.df['low']).rolling(window).std()

        return self

    def add_momentum_features(self):
        """
        Add additional momentum-based indicators like Stochastics, ADX, CCI, etc.
        (We already have RSI, MACD in add_base_timeframe_features, but you can unify them here if you prefer.)
        """
        self.df.sort_index(inplace=True)

        # Stochastic
        stoch = ta.stoch(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            k=14, d=3
        )
        self.df['stoch_k'] = stoch['STOCHk_14_3_3']
        self.df['stoch_d'] = stoch['STOCHd_14_3_3']

        # ADX (Average Directional Movement Index)
        adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
        self.df['adx'] = adx_data['ADX_14']
        self.df['diplus'] = adx_data['DMP_14']
        self.df['diminus'] = adx_data['DMN_14']

        # CCI (Commodity Channel Index)
        self.df['cci'] = ta.cci(self.df['high'], self.df['low'], self.df['close'], length=20)

        return self

    def add_volatility_features(self, window=20):
        """
        Add additional volatility-based features, e.g. historical volatility, Z-score of returns, etc.
        """
        self.df.sort_index(inplace=True)

        # Historical volatility (using close-to-close log returns)
        self.df['log_return'] = np.log(self.df['close'] / self.df['close'].shift(1))
        self.df['hist_vol'] = self.df['log_return'].rolling(window).std() * np.sqrt(252)  # Annualized approximation

        # Z-Score of Price Changes over 'window'
        rolling_mean = self.df['log_return'].rolling(window).mean()
        rolling_std = self.df['log_return'].rolling(window).std()
        self.df['zscore_return'] = (self.df['log_return'] - rolling_mean) / (rolling_std + 1e-9)

        return self

    def add_volume_features(self):
        """
        Add volume-based features if 'volume' is present.
        """
        if 'volume' not in self.df.columns:
            return self  # No volume data, skip

        self.df.sort_index(inplace=True)

        # On-Balance Volume
        self.df['obv'] = ta.obv(self.df['close'], self.df['volume'])

        # Volume spike detection:
        # Compare current volume to rolling average
        vol_mean = self.df['volume'].rolling(20).mean()
        vol_std = self.df['volume'].rolling(20).std()
        self.df['vol_spike'] = np.where(
            (self.df['volume'] > (vol_mean + 2 * vol_std)), 1.0, 0.0
        )

        # VWAP over the day (if you have daily or session logic, you'd groupby date)
        # Here, for a simpler approach, we do a rolling 1-day approximation:
        # This might not be perfectly accurate if you have continuous 24h data, but it’s a start.
        # Usually you’d reset VWAP each day or each session.
        typical_price = (self.df['high'] + self.df['low'] + self.df['close']) / 3.0
        self.df['cum_tp_vol'] = (typical_price * self.df['volume']).cumsum()
        self.df['cum_vol'] = self.df['volume'].cumsum()
        self.df['vwap'] = self.df['cum_tp_vol'] / (self.df['cum_vol'] + 1e-9)

        # Drop helper columns
        self.df.drop(['cum_tp_vol', 'cum_vol'], axis=1, inplace=True)

        return self

    def add_regime_features(self):
        """
        Classify each bar as 'trending' or 'ranging' (0 or 1) based on ADX, or volatility-based thresholds, etc.
        Keep it numeric for scaling (e.g., 0.0 or 1.0).
        """
        self.df.sort_index(inplace=True)

        # If ADX above 25 => trending, else ranging
        # (25 is just a typical reference threshold; pick whichever fits your strategy)
        if 'adx' not in self.df.columns:
            # If ADX wasn't yet computed, do it quickly
            adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
            self.df['adx'] = adx_data['ADX_14']

        self.df['regime_trend'] = np.where(self.df['adx'] >= 25, 1.0, 0.0)

        # Alternatively, you could have multiple columns for different regime classifications
        return self

    def add_time_features(self):
        """
        Extract cyclical or numeric time features (hour of day, day of week).
        We ensure they are numeric (float).
        """
        self.df.sort_index(inplace=True)

        # Because 'datetime' is now our index, we can access it via self.df.index
        # Create columns for hour, day of week
        self.df['hour'] = self.df.index.hour.astype(float)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(float)

        # Cyclical encoding for hour
        self.df['hour_sin'] = np.sin(2 * np.pi * self.df['hour'] / 24.0)
        self.df['hour_cos'] = np.cos(2 * np.pi * self.df['hour'] / 24.0)

        # Cyclical encoding for day_of_week
        self.df['dow_sin'] = np.sin(2 * np.pi * self.df['day_of_week'] / 7.0)
        self.df['dow_cos'] = np.cos(2 * np.pi * self.df['day_of_week'] / 7.0)

        # You can drop raw hour/day_of_week if you prefer only cyclical features
        # self.df.drop(['hour', 'day_of_week'], axis=1, inplace=True)

        return self

    def add_adaptive_targets_and_stops(self):
        """
        Instead of fixed (2 * ATR) / (1 * ATR), adapt based on current volatility or regime.
        For demonstration, we’ll say:
            - If regime is trending, target = 3 * ATR, stop = 1.5 * ATR
            - If regime is ranging, target = 1.5 * ATR, stop = 1 * ATR
        This is just an example logic.
        """
        self.df.sort_index(inplace=True)

        if 'atr_base' not in self.df.columns:
            self.df['atr_base'] = ta.atr(self.df['high'], self.df['low'], self.df['close'], length=14)

        # If we haven't computed 'regime_trend', do so
        if 'regime_trend' not in self.df.columns:
            self.add_regime_features()

        # Example adaptation
        is_trend = (self.df['regime_trend'] == 1.0)
        self.df['Target'] = np.where(is_trend, 3.0 * self.df['atr_base'], 1.5 * self.df['atr_base'])
        self.df['StopLoss'] = np.where(is_trend, 1.5 * self.df['atr_base'], 1.0 * self.df['atr_base'])

        return self

    def run_pipeline(self):
        """
        Run all steps except higher timeframe computations (now removed).
        You can rearrange the order as desired.
        """
        (self.preprocess_datetime()
             .clean_data()
             .add_base_timeframe_features()
             .add_candlestick_patterns()
             .add_swing_points()
             .add_range_breakout_features()
             .add_momentum_features()
             .add_volatility_features()
             .add_volume_features()            # only applies if 'volume' exists
             .add_regime_features()
             .add_time_features()
             .add_adaptive_targets_and_stops()
        )
        return self

    def get_processed_df(self):
        """
        Retrieve the final DataFrame (including any signals if label_signals() was called).
        We'll also ensure that all columns except 'Signal' are float with 2 decimal places.
        """
        # Drop rows that have NaN from lookbacks
        self.df.dropna(axis=0, how='any', inplace=True)

        # Drop 'Entry Price' and 'Exit Price' if you do not want them in final features
        if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)

        # Ensure 'Signal' is int; everything else float with 2 decimals
        if 'Signal' in self.df.columns:
            sig_col = self.df['Signal'].astype(int)
            self.df.drop(columns=['Signal'], inplace=True)
        else:
            sig_col = None

        # Convert all remaining columns to float
        for col in self.df.columns:
            self.df[col] = self.df[col].astype(float)

        # Round to 2 decimals
        self.df = self.df.round(2)

        # Reinsert Signal column if it exists
        if sig_col is not None:
            self.df['Signal'] = sig_col

        return self.df

nf_train_pipeline = FullFeaturePipeline(nf_train_df, two_interval_minutes)
nf_train_pipeline.run_pipeline()

nf_processed_train_df = nf_train_pipeline.get_processed_df()

bnf_train_pipeline = FullFeaturePipeline(bnf_train_df, two_interval_minutes)
bnf_train_pipeline.run_pipeline()

bnf_processed_train_df = bnf_train_pipeline.get_processed_df()

In [10]:
nf_processed_train_df

,open,high,low,close,rsi_base,macd,macd_h,macd_s,bb_lower,bb_mid,...,zscore_return,regime_trend,hour,day_of_week,hour_sin,hour_cos,dow_sin,dow_cos,Target,StopLoss
datetime,,,,,,,,,,,,,,,,,,,,,
2024-07-09 10:21:00,24344.65,24353.40,24339.95,24347.45,41.15,-2.16,-3.09,0.93,24345.67,24365.13,...,0.44,0.0,10.0,1.0,0.50,-0.87,0.78,0.62,20.74,13.83
2024-07-09 10:23:00,24346.60,24359.20,24345.35,24346.90,40.91,-3.11,-3.23,0.12,24343.24,24363.76,...,0.08,0.0,10.0,1.0,0.50,-0.87,0.78,0.62,20.74,13.83
2024-07-09 10:25:00,24345.45,24359.30,24345.45,24352.80,44.63,-3.35,-2.78,-0.57,24342.38,24362.59,...,0.68,1.0,10.0,1.0,0.50,-0.87,0.78,0.62,41.49,20.74
2024-07-09 10:27:00,24352.40,24361.40,24349.65,24359.50,48.59,-2.97,-1.92,-1.05,24342.29,24362.53,...,0.68,1.0,10.0,1.0,0.50,-0.87,0.78,0.62,41.01,20.50
2024-07-09 10:29:00,24358.70,24362.10,24353.25,24353.65,45.53,-3.10,-1.64,-1.46,24341.77,24362.30,...,-0.56,1.0,10.0,1.0,0.50,-0.87,0.78,0.62,39.90,19.95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-17 15:21:00,24748.20,24754.15,24747.10,24752.05,52.84,0.98,0.81,0.17,24732.60,24746.56,...,0.36,0.0,15.0,3.0,-0.71,-0.71,0.43,-0.90,14.69,9.79
2024-10-17 15:23:00,24750.50,24754.35,24747.80,24751.80,52.57,1.03,0.69,0.34,24734.38,24747.40,...,-0.24,0.0,15.0,3.0,-0.71,-0.71,0.43,-0.90,14.34,9.56
2024-10-17 15:25:00,24750.85,24760.25,24749.00,24750.45,51.08,0.96,0.49,0.47,24736.87,24748.25,...,-0.48,0.0,15.0,3.0,-0.71,-0.71,0.43,-0.90,14.52,9.68


In [11]:
bnf_processed_train_df

,open,high,low,close,rsi_base,macd,macd_h,macd_s,bb_lower,bb_mid,...,zscore_return,regime_trend,hour,day_of_week,hour_sin,hour_cos,dow_sin,dow_cos,Target,StopLoss
datetime,,,,,,,,,,,,,,,,,,,,,
2024-07-09 10:21:00,52486.05,52499.50,52476.25,52490.10,48.82,-9.70,8.11,-17.81,52425.33,52480.97,...,0.17,0.0,10.0,1.0,0.50,-0.87,0.78,0.62,61.23,40.82
2024-07-09 10:23:00,52489.80,52515.35,52480.75,52485.00,47.81,-9.26,6.83,-16.10,52427.61,52478.76,...,-0.11,0.0,10.0,1.0,0.50,-0.87,0.78,0.62,60.50,40.33
2024-07-09 10:25:00,52479.70,52505.40,52477.65,52491.70,49.30,-8.28,6.25,-14.53,52430.48,52476.98,...,0.32,0.0,10.0,1.0,0.50,-0.87,0.78,0.62,59.04,39.36
2024-07-09 10:27:00,52491.40,52511.80,52483.40,52501.25,51.41,-6.66,6.30,-12.96,52432.21,52479.09,...,0.35,0.0,10.0,1.0,0.50,-0.87,0.78,0.62,57.78,38.52
2024-07-09 10:29:00,52499.65,52514.00,52494.55,52505.70,52.41,-4.95,6.41,-11.36,52433.32,52481.11,...,0.11,0.0,10.0,1.0,0.50,-0.87,0.78,0.62,55.60,37.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-17 15:21:00,51308.60,51313.60,51286.55,51301.55,58.09,13.52,3.63,9.89,51241.05,51280.72,...,-0.96,0.0,15.0,3.0,-0.71,-0.71,0.43,-0.90,43.78,29.19
2024-10-17 15:23:00,51300.70,51309.65,51287.30,51301.55,58.09,13.20,2.65,10.56,51245.66,51283.38,...,-0.22,0.0,15.0,3.0,-0.71,-0.71,0.43,-0.90,43.05,28.70
2024-10-17 15:25:00,51300.80,51318.45,51285.75,51294.75,55.21,12.26,1.36,10.90,51247.44,51284.73,...,-0.69,0.0,15.0,3.0,-0.71,-0.71,0.43,-0.90,43.48,28.99


MAML RL Model

In [ ]:
################################################################################
# FULL JUPYTER NOTEBOOK:
# ADVANCED MAML AGENT WITH DEEP CUSTOM TRANSFORMER POLICY & HYPERPARAMETER TUNING
################################################################################

################################################################################
# CELL 1: Imports & Basic Setup
################################################################################

# You may need to install optuna for hyperparameter tuning:
# %pip install optuna

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import higher

# For hyperparameter search (Bayesian optimization)
import optuna

# We explicitly disable the newer scaled dot-product attention kernel usage:
torch.backends.cuda.flash_sdp_enabled = False
torch.backends.cuda.mem_efficient_sdp_enabled = False
torch.backends.cuda.math_sdp_enabled = False

# Check for CUDA/GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


################################################################################
# CELL 2: Base Configuration Dictionary (Will be Overwritten by Tuning)
################################################################################
# We define some baseline hyperparameters that can be overwritten dynamically
# during the hyperparameter tuning process. Some parameters remain fixed,
# while others (e.g. LR_META, LR_INNER, TRANSFORMER_MODEL_DIM, etc.)
# will be tuned in the objective function.
base_config = {
    "FEATURE_COLS": None,            # If None, auto-detect from DataFrame
    "EXCLUDE_COLS": ["Target", "StopLoss"],

    "INITIAL_CAPITAL": 10000.0,
    "QUANTITY": 1.0,
    "BROKERAGE_ENTRY": 20.0,
    "BROKERAGE_EXIT": 20.0,
    "FORCE_CLOSE_STEPS": 50,

    "DISCOUNT_FACTOR": 0.99,

    # These will be overridden by the hyperparameter tuner
    "LR_META": 1e-4,
    "LR_INNER": 1e-3,
    "META_TRAINING_EPOCHS": 10,
    "INNER_ADAPTATION_STEPS": 5,

    "MAX_STEPS_PER_EPISODE": 1000,

    # Custom Transformer-related parameters (could also be tuned)
    "SEQ_LEN": 10,
    "TRANSFORMER_MODEL_DIM": 128,
    "TRANSFORMER_FF_DIM": 256,
    "TRANSFORMER_NUM_HEADS": 4,
    "TRANSFORMER_NUM_LAYERS": 2,

    "ACTION_SPACE_SIZE": 4,  # 0=Hold, 1=Buy, 2=Sell, 3=Close

    # Model checkpoint path
    "MODEL_PATH": "maml_trader_transformer_tuned.pth"
}


################################################################################
# CELL 3: Helper to Setup Feature Columns
################################################################################
def setup_feature_columns(df, config):
    """
    If config["FEATURE_COLS"] is None, we dynamically detect columns
    except those in config["EXCLUDE_COLS"].
    """
    if config["FEATURE_COLS"] is None:
        exclude = set(config["EXCLUDE_COLS"])
        dynamic_cols = [c for c in df.columns if c not in exclude]
        config["FEATURE_COLS"] = dynamic_cols


################################################################################
# CELL 4: Custom Trading Environment with Step-by-Step Logging
################################################################################
class CustomTradingEnv:
    """
    Single-position environment with 4 actions: 0=Hold, 1=Buy, 2=Sell, 3=Close.
    Forced close after FORCE_CLOSE_STEPS. Reward = delta in capital.

    For the Transformer model, _get_observation() returns a sequence of length
    SEQ_LEN. If there are fewer than SEQ_LEN steps so far, it pads with zeros.

    This environment also logs step-by-step details in self.step_logs so that
    we can inspect them at any time.
    """
    def __init__(self, df, config):
        self.df = df.reset_index(drop=True)
        self.n_steps = len(self.df)
        self.config = config

        setup_feature_columns(self.df, self.config)
        self.feature_cols = self.config["FEATURE_COLS"]

        self.current_step = 0
        self.current_capital = self.config["INITIAL_CAPITAL"]
        self.position_type = None  # None, 'long', 'short'
        self.entry_price = 0.0
        self.target_price = 0.0
        self.stop_loss_price = 0.0
        self.position_open_step = 0

        # Logs for trades
        self.trades_log = []

        # Step-by-step logs:
        self.step_logs = []

        self.reset()

    def reset(self):
        self.current_step = 0
        self.current_capital = self.config["INITIAL_CAPITAL"]
        self.position_type = None
        self.entry_price = 0.0
        self.target_price = 0.0
        self.stop_loss_price = 0.0
        self.position_open_step = 0
        self.trades_log = []
        self.step_logs = []

        return self._get_observation()

    def _get_observation(self):
        """
        Returns a [SEQ_LEN x num_features] float32 array.
        If current_step < SEQ_LEN, pad with zeros at the front.
        """
        seq_len = self.config["SEQ_LEN"]
        start_idx = max(0, self.current_step - seq_len + 1)
        end_idx = self.current_step + 1
        subdf = self.df.iloc[start_idx:end_idx][self.feature_cols].values.astype(np.float32)

        if len(subdf) < seq_len:
            pad = np.zeros((seq_len - len(subdf), len(self.feature_cols)), dtype=np.float32)
            subdf = np.concatenate([pad, subdf], axis=0)

        return subdf

    def _close_position(self, reason):
        current_price = self.df['close'].iloc[self.current_step]
        if self.position_type == 'long':
            pnl = (current_price - self.entry_price) * self.config["QUANTITY"]
        else:  # short
            pnl = (self.entry_price - current_price) * self.config["QUANTITY"]

        self.current_capital += pnl
        self.current_capital -= self.config["BROKERAGE_EXIT"]

        self.trades_log.append({
            'step_exit': self.current_step,
            'position_type': self.position_type,
            'entry_price': self.entry_price,
            'exit_price': current_price,
            'target_price': self.target_price,
            'stop_loss_price': self.stop_loss_price,
            'close_reason': reason,
            'pnl': pnl,
            'capital_after': self.current_capital
        })

        # Reset position
        self.position_type = None
        self.entry_price = 0.0
        self.target_price = 0.0
        self.stop_loss_price = 0.0
        self.position_open_step = 0

    def step(self, action):
        done = False
        old_capital = self.current_capital

        # If there's an open position, check SL or forced-close
        if self.position_type is not None and self.current_step < self.n_steps:
            current_price = self.df['close'].iloc[self.current_step]

            # StopLoss check
            if self.position_type == 'long' and current_price <= self.stop_loss_price:
                self._close_position(reason='SL_hit')
            elif self.position_type == 'short' and current_price >= self.stop_loss_price:
                self._close_position(reason='SL_hit')

            # Force close
            if self.position_type is not None:
                if (self.current_step - self.position_open_step) >= self.config["FORCE_CLOSE_STEPS"]:
                    self._close_position(reason='Force_close')

        # If no open position, possible new open
        if self.position_type is None:
            if action == 1 and self.current_step < self.n_steps:  # Buy
                current_price = self.df['close'].iloc[self.current_step]
                self.current_capital -= self.config["BROKERAGE_ENTRY"]
                self.position_type = 'long'

                t = self.df['Target'].iloc[self.current_step]
                sl = self.df['StopLoss'].iloc[self.current_step]
                self.entry_price = current_price
                self.target_price = current_price + t
                self.stop_loss_price = current_price - sl
                self.position_open_step = self.current_step

            elif action == 2 and self.current_step < self.n_steps:  # Sell
                current_price = self.df['close'].iloc[self.current_step]
                self.current_capital -= self.config["BROKERAGE_ENTRY"]
                self.position_type = 'short'

                t = self.df['Target'].iloc[self.current_step]
                sl = self.df['StopLoss'].iloc[self.current_step]
                self.entry_price = current_price
                self.target_price = current_price - t
                self.stop_loss_price = current_price + sl
                self.position_open_step = self.current_step
        else:
            # If there's already a position, user can choose action=3 to manually close it
            if action == 3:
                self._close_position(reason='Manual_close')

        # Move to next step
        self.current_step += 1
        if self.current_step >= self.n_steps:
            # End of data, close any open position
            if self.position_type is not None:
                self._close_position(reason='End_of_data')
            done = True

        # Compute reward
        reward = self.current_capital - old_capital

        # Log the step
        self.step_logs.append({
            'step': self.current_step,
            'action': action,
            'reward': reward,
            'capital': self.current_capital,
            'position_type': self.position_type,
        })

        next_obs = self._get_observation()
        return next_obs, reward, done

    def get_trade_log_df(self):
        if len(self.trades_log) == 0:
            return pd.DataFrame(columns=[
                'step_exit','position_type','entry_price','exit_price',
                'target_price','stop_loss_price','close_reason','pnl','capital_after'
            ])
        return pd.DataFrame(self.trades_log)

    def get_step_log_df(self):
        if len(self.step_logs) == 0:
            return pd.DataFrame(columns=['step', 'action', 'reward', 'capital', 'position_type'])
        return pd.DataFrame(self.step_logs)


################################################################################
# CELL 5: Custom Transformer Modules
# (We do not rely on torch.nn.Transformer for demonstration.)
################################################################################

def scaled_dot_product_attention(q, k, v, mask=None):
    """
    Manually compute Scaled Dot-Product Attention:
    - q, k, v: shape (batch_size, num_heads, seq_len, head_dim)
    - mask: optional (batch_size, 1, seq_len, seq_len)
    Returns context and attention weights for each head.
    """
    d_k = q.shape[-1]
    scores = torch.matmul(q, k.transpose(-2, -1)) / (d_k ** 0.5)  # (B, nH, seq_len, seq_len)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attn = torch.softmax(scores, dim=-1)  # (B, nH, seq_len, seq_len)
    context = torch.matmul(attn, v)       # (B, nH, seq_len, head_dim)
    return context, attn


class CustomMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (B, seq_len, d_model)
        B, seq_len, _ = x.size()

        # project inputs
        q = self.q_linear(x)  # (B, seq_len, d_model)
        k = self.k_linear(x)
        v = self.v_linear(x)

        # reshape for multi-head
        q = q.view(B, seq_len, self.num_heads, self.head_dim).transpose(1, 2)  # (B, nH, seq_len, head_dim)
        k = k.view(B, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # scaled dot product attention
        context, _ = scaled_dot_product_attention(q, k, v, mask=None)
        # context: (B, nH, seq_len, head_dim)

        # swap back
        context = context.transpose(1, 2).contiguous().view(B, seq_len, self.d_model)  # (B, seq_len, d_model)

        # final linear + dropout
        out = self.out_proj(context)  # (B, seq_len, d_model)
        out = self.dropout(out)

        # residual + layer norm
        out = self.layer_norm(x + out)
        return out


class CustomFeedForward(nn.Module):
    def __init__(self, d_model, dim_feedforward, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (B, seq_len, d_model)
        residual = x
        out = self.linear1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.linear2(out)
        out = self.dropout(out)

        out = self.layer_norm(residual + out)
        return out


class CustomTransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dim_feedforward, dropout=0.1):
        super().__init__()
        self.mha = CustomMultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.ff = CustomFeedForward(d_model, dim_feedforward, dropout=dropout)

    def forward(self, x):
        out = self.mha(x)
        out = self.ff(out)
        return out


class CustomTransformerEncoder(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, num_layers, dropout=0.1):
        super().__init__()
        layers = []
        for _ in range(num_layers):
            layers.append(CustomTransformerBlock(d_model, num_heads, ff_dim, dropout=dropout))
        self.blocks = nn.Sequential(*layers)

    def forward(self, x):
        return self.blocks(x)  # (B, seq_len, d_model)


################################################################################
# CELL 6: Custom Transformer Policy (Deep "GPT-like" Blocks)
################################################################################
class CustomTransformerPolicy(nn.Module):
    """
    A custom Transformer-based policy using MHA+FF blocks
    with multiple layers to resemble a deeper "GPT-style" approach.
    """
    def __init__(self, feature_dim, seq_len, d_model, num_heads, ff_dim, num_layers, num_actions):
        super().__init__()
        self.seq_len = seq_len
        self.d_model = d_model

        # Project input features to model dimension
        self.input_proj = nn.Linear(feature_dim, d_model)

        # Learned positional encoding
        self.pos_encoding = nn.Parameter(torch.zeros(1, seq_len, d_model))

        # Stacked custom transformer encoder
        self.encoder = CustomTransformerEncoder(
            d_model=d_model,
            num_heads=num_heads,
            ff_dim=ff_dim,
            num_layers=num_layers,
            dropout=0.1
        )
        # Final linear layer for action logits
        self.output_layer = nn.Linear(d_model, num_actions)

    def forward(self, x):
        """
        x: (batch_size, seq_len, feature_dim)
        returns logits: (batch_size, num_actions)
        """
        # Project features
        x = self.input_proj(x)  # (B, seq_len, d_model)
        # Add (learned) positional encoding
        x = x + self.pos_encoding[:, :x.size(1), :]

        # Pass through custom transformer blocks
        x = self.encoder(x)  # (B, seq_len, d_model)

        # Take the last timestep's output (like GPT next-token approach)
        final_timestep = x[:, -1, :]  # (B, d_model)

        # Map to action logits
        logits = self.output_layer(final_timestep)  # (B, num_actions)
        return logits


################################################################################
# CELL 7: MAML Trader with "higher" for Meta-Learning + Step Logging
################################################################################
class MAMLTrader:
    def __init__(self, config):
        self.config = config
        self.gamma = config["DISCOUNT_FACTOR"]
        self.policy = None
        self.meta_optimizer = None

        # We will store step-logs from rollouts in case the user wants
        # to inspect them after each training environment:
        self.rollout_logs = []

    def initialize_policy(self, input_dim):
        seq_len = self.config["SEQ_LEN"]
        d_model = self.config["TRANSFORMER_MODEL_DIM"]
        ff_dim = self.config["TRANSFORMER_FF_DIM"]
        num_heads = self.config["TRANSFORMER_NUM_HEADS"]
        num_layers = self.config["TRANSFORMER_NUM_LAYERS"]
        num_actions = self.config["ACTION_SPACE_SIZE"]

        self.policy = CustomTransformerPolicy(
            feature_dim=input_dim,
            seq_len=seq_len,
            d_model=d_model,
            num_heads=num_heads,
            ff_dim=ff_dim,
            num_layers=num_layers,
            num_actions=num_actions
        ).to(device)

        self.meta_optimizer = optim.Adam(self.policy.parameters(), lr=self.config["LR_META"])

    def load_model_if_exists(self):
        model_path = self.config["MODEL_PATH"]
        if os.path.exists(model_path):
            checkpoint = torch.load(model_path, map_location=device)
            self.policy.load_state_dict(checkpoint)
            self.policy.to(device)
            print(f"Loaded existing model from: {model_path}")
        else:
            print("No existing model found. Starting fresh.")

    def save_model(self):
        model_path = self.config["MODEL_PATH"]
        torch.save(self.policy.state_dict(), model_path)
        print(f"Model saved to: {model_path}")

    def _select_action(self, state, policy_params=None):
        """
        state: (SEQ_LEN, feature_dim) in numpy.
        We'll add a batch dimension => (1, SEQ_LEN, feature_dim).
        """
        state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        if policy_params is None:
            logits = self.policy(state_t)
        else:
            logits = policy_params(state_t)

        probs = torch.softmax(logits, dim=-1)
        action = torch.multinomial(probs, 1).item()
        return action

    def _compute_discounted_returns(self, rewards):
        G = 0
        returns = []
        for r in reversed(rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
        return np.array(returns, dtype=np.float32)

    def _rollout_single_env(self, env, fmodel):
        """
        Rolls out a single episode in the environment using fmodel (fast weights).
        Returns (states, actions, discounted_returns).
        Also logs each step in the environment's step_logs for later inspection.
        """
        states_buffer, actions_buffer, rewards_buffer = [], [], []
        state = env.reset()
        done = False
        steps = 0

        while not done and steps < self.config["MAX_STEPS_PER_EPISODE"]:
            action = self._select_action(state, policy_params=fmodel)
            next_state, reward, done = env.step(action)

            states_buffer.append(state)
            actions_buffer.append(action)
            rewards_buffer.append(reward)

            state = next_state
            steps += 1

        returns_buffer = self._compute_discounted_returns(rewards_buffer)

        # Save environment step logs to our global rollout logs
        step_df = env.get_step_log_df()
        if not step_df.empty:
            self.rollout_logs.append(step_df.copy())

        return np.array(states_buffer), np.array(actions_buffer), returns_buffer

    def _compute_loss(self, fmodel, states, actions, returns):
        """
        Standard REINFORCE loss:
        Negative log likelihood times discounted returns
        """
        states_t = torch.tensor(states, dtype=torch.float32, device=device)   # (B, seq_len, feature_dim)
        actions_t = torch.tensor(actions, dtype=torch.long, device=device)    # (B,)
        returns_t = torch.tensor(returns, dtype=torch.float32, device=device) # (B,)

        logits = fmodel(states_t)  # (B, num_actions)
        log_probs = torch.log_softmax(logits, dim=-1)
        chosen_log_probs = log_probs[range(len(actions_t)), actions_t]

        loss = -torch.mean(chosen_log_probs * returns_t)
        return loss

    def meta_train_offline(self, data_dfs):
        """
        Perform offline meta-training using multiple tasks (data_dfs).
        """
        # Setup for first dataset
        temp_env = CustomTradingEnv(data_dfs[0], self.config)
        input_dim = len(temp_env.feature_cols)

        if self.policy is None:
            self.initialize_policy(input_dim)

        self.load_model_if_exists()
        print("Starting Offline Meta-Training...")

        for epoch in range(self.config["META_TRAINING_EPOCHS"]):
            meta_loss_val = 0.0

            # Clear rollout logs each epoch so we only store this epoch's logs
            self.rollout_logs = []

            for df in data_dfs:
                env = CustomTradingEnv(df, self.config)
                inner_opt = optim.SGD(self.policy.parameters(), lr=self.config["LR_INNER"])

                with higher.innerloop_ctx(self.policy, inner_opt, copy_initial_weights=False) as (fmodel, diffopt):
                    # 1) Rollout for initial adaptation
                    s_buf1, a_buf1, r_buf1 = self._rollout_single_env(env, fmodel)
                    loss_task = self._compute_loss(fmodel, s_buf1, a_buf1, r_buf1)

                    for _ in range(self.config["INNER_ADAPTATION_STEPS"]):
                        diffopt.step(loss_task)

                    # 2) Meta-update using second rollout
                    s_buf2, a_buf2, r_buf2 = self._rollout_single_env(env, fmodel)
                    loss_meta = self._compute_loss(fmodel, s_buf2, a_buf2, r_buf2)
                    meta_loss_val += loss_meta.item()

                    loss_meta.backward()

            # Outer step
            self.meta_optimizer.step()
            self.meta_optimizer.zero_grad()

            meta_loss_val /= len(data_dfs)
            print(f"Epoch [{epoch+1}/{self.config['META_TRAINING_EPOCHS']}], Meta-Loss: {meta_loss_val:.6f}")

        self.save_model()
        print("Offline Meta-Training complete.\n")

    def online_train_step(self, new_data_df):
        """
        Adapt to new data (single environment) using a MAML-like approach.
        """
        env = CustomTradingEnv(new_data_df, self.config)
        input_dim = len(env.feature_cols)

        if self.policy is None:
            self.initialize_policy(input_dim)

        self.load_model_if_exists()

        inner_opt = optim.SGD(self.policy.parameters(), lr=self.config["LR_INNER"])

        with higher.innerloop_ctx(self.policy, inner_opt, copy_initial_weights=False) as (fmodel, diffopt):
            # 1) Rollout for initial adaptation
            s_buf1, a_buf1, r_buf1 = self._rollout_single_env(env, fmodel)
            loss_task = self._compute_loss(fmodel, s_buf1, a_buf1, r_buf1)

            for _ in range(self.config["INNER_ADAPTATION_STEPS"]):
                diffopt.step(loss_task)

            # 2) Meta-update using second rollout
            s_buf2, a_buf2, r_buf2 = self._rollout_single_env(env, fmodel)
            loss_meta = self._compute_loss(fmodel, s_buf2, a_buf2, r_buf2)
            loss_meta.backward()

        self.meta_optimizer.step()
        self.meta_optimizer.zero_grad()
        self.save_model()
        return loss_meta.item()


################################################################################
# CELL 8: Utility for Final Results
################################################################################
def generate_final_results_df(trades_log_df):
    """
    Summarize trades: PnL, reason for close, overall WinRate, etc.
    """
    if trades_log_df.empty:
        return pd.DataFrame(columns=[
            'PositionType','EntryPrice','TargetPrice','StopLossPrice',
            'CloseReason','TargetHit','SLHit','PnL','Capital','WinRate(%)'
        ])

    results = []
    for _, row in trades_log_df.iterrows():
        position_type = row['position_type']
        entry_price = row['entry_price']
        exit_price = row['exit_price']
        target_price = row['target_price']
        stop_loss_price = row['stop_loss_price']
        pnl = row['pnl']
        capital_after = row['capital_after']
        close_reason = row['close_reason']

        target_hit = False
        sl_hit = False
        if close_reason == 'SL_hit':
            sl_hit = True
        else:
            if position_type == 'long' and exit_price >= target_price:
                target_hit = True
            elif position_type == 'short' and exit_price <= target_price:
                target_hit = True

        results.append({
            'PositionType': position_type,
            'EntryPrice': entry_price,
            'TargetPrice': target_price,
            'StopLossPrice': stop_loss_price,
            'CloseReason': close_reason,
            'TargetHit': target_hit,
            'SLHit': sl_hit,
            'PnL': pnl,
            'Capital': capital_after
        })

    final_df = pd.DataFrame(results)
    total_trades = len(final_df)
    wins = len(final_df[final_df['PnL'] > 0])
    win_rate = (wins / total_trades * 100.0) if total_trades > 0 else 0.0
    final_df['WinRate(%)'] = win_rate
    return final_df


################################################################################
# CELL 9: Hyperparameter Tuning with Optuna (Dynamic Hyperparameter Search)
################################################################################
"""
Below is an example advanced approach using Bayesian optimization with Optuna.
We define an objective function that:
1) Samples hyperparameters (LR_META, LR_INNER, TRANSFORMER_MODEL_DIM, etc.)
2) Partially trains the MAML Trader for a few epochs on a subset of data
3) Evaluates the average meta-loss over the last epoch
4) Returns that meta-loss to be minimized

In practice, you might want to hold out a validation dataset to measure
the performance. For brevity, we just measure the meta-loss on the training tasks.
"""

def hyperparameter_tuning(train_dfs, n_trials=10):
    """
    :param train_dfs: List of DataFrames for meta-training
    :param n_trials: Number of Optuna trials
    :return: best_params (dict) from the study
    """

    def objective(trial):
        # Create a copy of base_config so we can modify it
        config_tuned = dict(base_config)

        # Sample hyperparameters
        config_tuned["LR_META"] = trial.suggest_loguniform("LR_META", 1e-5, 1e-3)
        config_tuned["LR_INNER"] = trial.suggest_loguniform("LR_INNER", 1e-4, 1e-2)
        config_tuned["META_TRAINING_EPOCHS"] = trial.suggest_int("META_TRAINING_EPOCHS", 5, 15)
        config_tuned["INNER_ADAPTATION_STEPS"] = trial.suggest_int("INNER_ADAPTATION_STEPS", 2, 6)

        # Transformer architecture sampling
        config_tuned["TRANSFORMER_MODEL_DIM"] = trial.suggest_categorical("TRANSFORMER_MODEL_DIM", [128, 256])
        config_tuned["TRANSFORMER_FF_DIM"] = trial.suggest_categorical("TRANSFORMER_FF_DIM", [256, 512])
        config_tuned["TRANSFORMER_NUM_HEADS"] = trial.suggest_categorical("TRANSFORMER_NUM_HEADS", [4, 8])
        config_tuned["TRANSFORMER_NUM_LAYERS"] = trial.suggest_int("TRANSFORMER_NUM_LAYERS", 2, 6)

        # We do a partial training for each trial
        trader_temp = MAMLTrader(config_tuned)

        # We'll just do a quick pass and track the final meta-loss
        # to see how well the hyperparameters performed
        # (In a real scenario, you'd split train/val or measure final PnL, etc.)
        try:
            # We'll do a short meta-training (the final meta-loss is our objective)
            # For speed, we might limit the training data or reduce MAX_STEPS_PER_EPISODE
            # in the config, but let's do it as is for demonstration.
            trader_temp.meta_train_offline(train_dfs)

            # We do not have a separate validation in this minimal example,
            # so let's assume the final average meta-loss from the last epoch
            # is what we want to minimize. We'll retrieve it from the console logs
            # or store it in the class. For simplicity, store it in a variable:
            # -> We'll track the "meta_loss_val" in meta_train_offline if we modify it to return.
            # Instead, let's do a pseudo approach: we re-run a single environment to get a loss measure.

            # We'll just do a single rollout & measure loss
            single_env = CustomTradingEnv(train_dfs[0], config_tuned)
            input_dim = len(single_env.feature_cols)
            # If the policy wasn't re-initialized inside meta_train_offline, it's loaded
            # so we can evaluate quickly:
            # We'll just run a single rollout with the final weights:
            states, actions, returns = _evaluate_final_loss(trader_temp, single_env)
            # compute a final loss:
            final_loss = trader_temp._compute_loss(trader_temp.policy, states, actions, returns).item()

        except Exception as e:
            print("Exception in objective trial:", e)
            # If training fails, return a large loss so we avoid these params
            final_loss = 1e9

        return final_loss

    def _evaluate_final_loss(trader_temp, env):
        """
        Simple helper to do a single rollout & gather data.
        """
        s_buf, a_buf, r_buf = [], [], []
        state = env.reset()
        done = False
        steps = 0
        while not done and steps < trader_temp.config["MAX_STEPS_PER_EPISODE"]:
            action = trader_temp._select_action(state)
            next_state, reward, done = env.step(action)
            s_buf.append(state)
            a_buf.append(action)
            r_buf.append(reward)
            state = next_state
            steps += 1
        returns_buf = trader_temp._compute_discounted_returns(r_buf)
        return np.array(s_buf), np.array(a_buf), returns_buf

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    print("Best trial:")
    print("  Value (final loss):", study.best_value)
    print("  Params:", study.best_params)
    return study.best_params


################################################################################
# CELL 10: Example Usage - Combining Tuning + Training + Evaluation
################################################################################

# Assume you already have your training DataFrames:
# nf_processed_train_df, bnf_processed_train_df, etc.
# For demonstration, we use placeholders "train_dfs" as a list.
# Make sure each DF has columns ["close", "Target", "StopLoss"] plus features.

try:
    train_dfs = [nf_processed_train_df, bnf_processed_train_df]  # user-provided

    # 1) Dynamic Hyperparameter Tuning (Bayesian Optimization)
    best_hparams = hyperparameter_tuning(train_dfs, n_trials=10)

    # 2) Update base_config with best hyperparams and do final training
    tuned_config = dict(base_config)
    tuned_config.update(best_hparams)
    print("Using Tuned Hyperparameters:", tuned_config)

    # 3) Final training with best hyperparams
    final_trader = MAMLTrader(tuned_config)
    final_trader.meta_train_offline(train_dfs)

    # 4) (Optional) Evaluate on a test dataset (assume "test_df" is loaded)
    # test_df = ...
    # test_env = CustomTradingEnv(test_df, tuned_config)
    # state = test_env.reset()
    # done = False
    # while not done:
    #     action = final_trader._select_action(state)
    #     next_state, reward, done = test_env.step(action)
    #     state = next_state

    # trades_log_df = test_env.get_trade_log_df()
    # results_df = generate_final_results_df(trades_log_df)
    # display(results_df)

except NameError:
    # If nf_processed_train_df or bnf_processed_train_df are not defined
    # (since they're placeholders), handle it gracefully:
    print("Please define your train DataFrames (nf_processed_train_df, bnf_processed_train_df) before running.")
    print("This notebook is set up to demonstrate the full pipeline with user-provided data.")

[I 2025-01-25 03:46:56,262] A new study created in memory with name: no-name-547a5aae-08dd-4a6f-bf22-603717449aaa
<ipython-input-12-6daa85d36e7c>:724: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  config_tuned["LR_META"] = trial.suggest_loguniform("LR_META", 1e-5, 1e-3)
<ipython-input-12-6daa85d36e7c>:725: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  config_tuned["LR_INNER"] = trial.suggest_loguniform("LR_INNER", 1e-4, 1e-2)


No existing model found. Starting fresh.
Starting Offline Meta-Training...
Epoch [1/9], Meta-Loss: -573.070282


In [ ]:
env = CustomTradingEnv(test_df, config)
state = env.reset()
done = False
while not done:
    action = trader._select_action(state)
    next_state, reward, done = env.step(action)
    state = next_state

trades_log_df = env.get_trade_log_df()
results_df = generate_final_results_df(trades_log_df)
display(results_df)

# If you want to see the step-by-step logs:
step_logs_df = env.get_step_log_df()
display(step_logs_df)

In [ ]:
def get_sleep_time(interval_minutes, market_start_hour=9, market_start_minute=15, market_close_hour=15, market_close_minute=0):
    # Get current time in IST
    now = datetime.now(pytz.utc).astimezone(ist_timezone)

    # Define the market start and close times in IST for today
    market_start_time = now.replace(hour=market_start_hour, minute=market_start_minute, second=0, microsecond=0)
    market_close_time = now.replace(hour=market_close_hour, minute=market_close_minute, second=0, microsecond=0)

    if now < market_start_time:
        # If current time is before market starts, set next_run_time to market start time
        next_run_time = market_start_time
    elif now > market_close_time:
        # If current time is after market close, calculate the time until the next market open
        next_market_start_time = market_start_time + timedelta(days=1)
        next_run_time = next_market_start_time
    else:
        # Calculate the minutes since the market start time
        minutes_since_market_start = (now - market_start_time).total_seconds() // 60
        # Calculate the number of minutes to the next interval boundary
        minutes_to_next_interval = interval_minutes - (minutes_since_market_start % interval_minutes)
        # Calculate the next run time by adding these minutes to the current time
        next_run_time = (now + timedelta(minutes=minutes_to_next_interval)).replace(second=0, microsecond=0)

    # Calculate the sleep time in seconds
    sleep_time = (next_run_time - now).total_seconds()
    return sleep_time

In [ ]:
def fetch_option_chain():
    while True:
        try:
            data = {
                "symbol": index_symbol,
                "strikecount": 2,
                "timestamp": ""
            }
            response = fyers.optionchain(data=data)

            if response is not None:
                return response
        except Exception as e:
            print(f"Error fetching Option Chain: {e}")
            time.sleep(active_order_sleep)

index_oc= fetch_option_chain()

pd.DataFrame(index_oc['data']['optionsChain'])

In [ ]:
def assign_ce_pe_option_symbols():
    symbol_oc = fetch_option_chain()

    if symbol_oc != None:
        # Convert the response data into a DataFrame
        oc_df = pd.DataFrame(symbol_oc['data']['optionsChain'])

        # Find the first 'CE' symbol from the top
        first_ce_symbol = None
        for index, row in oc_df.iterrows():
            if row['option_type'] == 'CE':
                first_ce_symbol = row['symbol']
                first_ce_strike = row['strike_price']
                break

        # Find the first 'PE' symbol from the bottom
        first_pe_symbol = None
        for index, row in oc_df[::-1].iterrows():  # Iterate in reverse
            if row['option_type'] == 'PE':
                first_pe_symbol = row['symbol']
                first_pe_strike = row['strike_price']
                break

        return first_ce_symbol, first_pe_symbol, first_ce_strike, first_pe_strike

In [ ]:
def onmessage_ce(ce_message):
    global ce_ltp, index_ltp, unsubscribe_done
    try:
        if ce_message['symbol'] == ce_symbol:
            if "ltp" in ce_message:
                ce_ltp = ce_message["ltp"]
                ce_ltp = float(ce_ltp)

        elif ce_message['symbol'] == index_symbol:
            if "ltp" in ce_message:
                index_ltp = ce_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [ce_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {ce_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessageCE): {e}")


def onerror_ce(message):
    print("CE Error:", message)


def onclose_ce(message):
    print("CE Connection closed:", message)


def onopen_ce():

    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [ce_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def ce_buy_sell_ltp():
    global buy_sell_checked, ce_symbol, ce_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching CE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if ce_symbol is not None and ce_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                ce_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_ce,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_ce,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_ce,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_ce             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                ce_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def onmessage_pe(pe_message):
    global pe_ltp, index_ltp, unsubscribe_done
    try:
        if pe_message['symbol'] == pe_symbol:
            if "ltp" in pe_message:
                pe_ltp = pe_message["ltp"]
                pe_ltp = float(pe_ltp)

        elif pe_message['symbol'] == index_symbol:
            if "ltp" in pe_message:
                index_ltp = pe_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [pe_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {pe_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessagePE): {e}")

def onerror_pe(message):
    print("PE Error:", message)


def onclose_pe(message):
    print("PE Connection closed:", message)


def onopen_pe():
    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [pe_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def pe_buy_sell_ltp():
    global buy_sell_checked, pe_symbol, pe_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching PE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if pe_symbol is not None and pe_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                pe_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_pe,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_pe,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_pe,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_pe             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                pe_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def place_order(symbol):
    try:
        market_order_data = {
            "symbol": symbol,
            "qty": int(quantity),
            "type": 2,  # Market Order
            "side": 1,
            "productType": "INTRADAY",
            "limitPrice": 0,
            "stopPrice": 0,
            "validity": "DAY",
            "disclosedQty": 0,
            "offlineOrder":False
        }

        market_order_entry = fyers.place_order(data=market_order_data)

        if "id" in market_order_entry:
            market_order_id = market_order_entry["id"]
            market_order_message = market_order_entry["message"]
            print(f"{market_order_message}")

    except Exception as e:
        print(f"Error placing orders: {str(e)}")

In [ ]:
def trail_order(symbol, stoploss):
    while True:
        try:
            stoploss = int(stoploss)
            pending_order = fyers.orderbook()

            matching_orders = [order for order in pending_order["orderBook"] if order["status"] == 6]

            modified_orders = 0

            for order in matching_orders:
                if order['symbol'] == symbol:
                    pending_order_id = order['id']
                    pending_order_side = order['side']
                    pending_order_side = int(pending_order_side)

                    if pending_order_side != 1:
                        data = {
                            "id": pending_order_id,
                            "type": 4,
                            "limitPrice": stoploss - 1,
                            "stopPrice": stoploss
                        }

                        modify = fyers.modify_order(data=data)
                        trail_message = modify["message"]
                        print(f"{trail_message}")

                        if trail_message == "Successfully modified order":
                            modified_orders += 1

            # Check if all matching orders are successfully modified
            if modified_orders == len(matching_orders):
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print("Error modifying order:" + str(e))

In [ ]:
def exit_active_order(symbol):
    while True:
        try:
            data = {
                "id":f"{symbol}-INTRADAY"
            }

            exit_response = fyers.exit_positions(data=data)

            if ["message"] in exit_response:
                print(exit_response["message"])
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print(f"Error exiting Order: {e}")

In [ ]:
def reset_flags():
    global active_order, buy_sell_checked

    active_order = False
    buy_sell_checked = False

In [ ]:
# Function to save profits and losses
def save_overall(overall_win, overall_loss, capital):
    trade_type = {
        "overall_win": overall_win,
        "overall_loss": overall_loss,
        "capital": capital
    }

    with open("trade_data.json", "w") as file:
        json.dump(trade_type, file)


# Function to load wins and losses
def load_overall():
    try:
        with open('trade_data.json') as file:
            return json.load(file)
    except FileNotFoundError:
        return None

In [ ]:
def handle_active_ce_order():
    def ce_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, ce_ltp, index_ltp, ce_strike, ce_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        ce_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if ce_ltp != 0 and index_ltp != 0:
                    ce_ltp_array.append(ce_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)


                    if index_ltp <= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(ce_symbol)

                        points = int(ce_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("CE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp >= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                            trailing_sl_inside = int(ce_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp - stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                    else:
                        if (index_ltp - prev_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(ce_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp - trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("CE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=ce_order_loop()).start()

In [ ]:
def handle_active_pe_order():
    def pe_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, pe_ltp, index_ltp, pe_strike, pe_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        pe_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if pe_ltp != 0 and index_ltp != 0:
                    pe_ltp_array.append(pe_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)

                    if index_ltp >= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(pe_symbol)

                        points = int(pe_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("PE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp_array}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp <= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                            trailing_sl_inside = int(pe_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp + stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                    else:
                        if (prev_ltp - index_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(pe_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp + trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("PE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=pe_order_loop()).start()

In [ ]:
def ce_entry():
    threading.Thread(target=ce_buy_sell_ltp).start()

    def ce_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if ce_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = ce_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp + target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp - trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(ce_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_ce_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=ce_entry_thread()).start()

In [ ]:
def pe_entry():
    threading.Thread(target=pe_buy_sell_ltp).start()

    def pe_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if pe_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = pe_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp - target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp + trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(pe_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_pe_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=pe_entry_thread()).start()

In [ ]:
def market_entry_exit_logic(action, actual_closing_price, final_df):
    global sl_hit_condition, unsubscribe_done, ce_ltp, pe_ltp, index_ltp, fixed_ltp, fixed_index_ltp, prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, ce_strike, pe_strike, ce_symbol, pe_symbol

    ce_ltp = 0
    pe_ltp  =0
    index_ltp = 0
    fixed_ltp = 0
    fixed_index_ltp = 0
    prev_ltp = 0
    target_inside = 0
    target_index_inside = 0
    trailing_sl_inside = 0
    trailing_index_inside = 0
    ce_strike = None
    pe_strike = None
    ce_symbol = None
    pe_symbol = None

    final_current_time = final_df.index[-1].time()
    print(final_current_time)

    # Ensure trading occurs only during market hours
    if final_current_time >= dt_time(9, (15 + interval_minutes)) and final_current_time <= dt_time(18, 0):
        if action == 1 and not active_order:  # Buy action
            print("Buy action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making CE Position at {actual_closing_price}")
            ce_entry()  # Execute CE entry logic
        elif action == 2 and not active_order:  # Sell action
            print("Sell action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making PE Position at {actual_closing_price}")
            pe_entry()  # Execute PE entry logic
        elif action == 0:
            print("Hold action detected by PPO model. No trade executed.")

In [ ]:
def fetch_and_prepare_final_data():
    final_df = fetch_candle_data(10)

    final_data_processor = DataProcessor(final_df, live_processing=True)

    final_processed_df = final_data_processor.process().df

    return final_processed_df

In [ ]:
def find_local_extrema(df):
    order=5
    atr_multiplier=1.5
    min_distance=5

    # Find local maxima and minima
    local_max = argrelextrema(df['high'].values, np.greater_equal, order=order)[0]
    local_min = argrelextrema(df['low'].values, np.less_equal, order=order)[0]

    # Calculate the threshold based on ATR
    threshold = df['atr'] * atr_multiplier

    # Filter by significance
    significant_max = []
    significant_min = []

    for idx in local_max:
        if idx > order and idx < len(df) - order:
            high = df['high'].iloc[idx]
            if significant_min:
                low = df['low'].iloc[significant_min[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_max.append(idx)
            else:
                significant_max.append(idx)

    for idx in local_min:
        if idx > order and idx < len(df) - order:
            low = df['low'].iloc[idx]
            if significant_max:
                high = df['high'].iloc[significant_max[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_min.append(idx)
            else:
                significant_min.append(idx)

    # Ensure minimum distance
    def filter_by_distance(points, min_distance):
        filtered_points = []
        for i in range(len(points)):
            if not filtered_points or (points[i] - filtered_points[-1]) > min_distance:
                filtered_points.append(points[i])
        return filtered_points

    significant_max = filter_by_distance(significant_max, min_distance)
    significant_min = filter_by_distance(significant_min, min_distance)

    return significant_max, significant_min

In [ ]:
def get_trendline(df, point1, point2, kind='high'):
    x = [point1, point2]
    if kind == 'high':
        y = df['high'].values[x]
    else:
        y = df['low'].values[x]
    coeffs = np.polyfit(x, y, 1)
    trendline = np.polyval(coeffs, range(len(df)))
    return trendline

In [ ]:
# Online training configuration
online_learning_steps = 1000  # Steps for each retraining
retrain_interval_minutes = 60  # Retrain the model every 60 minutes

start_time = datetime.now()

while True:
    clear_output(wait=True)

    num_candles = 100

    final_df = fetch_and_prepare_final_data()

    final_df = final_df.iloc[:-1]

    target = final_df['Target'].iloc[-1]
    trailing_sl = final_df['Stop Loss'].iloc[-1]

    # Initialize the TradingEnvironment with the latest data
    live_env = create_env(final_df, config)
    obs = live_env.reset()

    # Use PPO model to predict action for the latest data point
    action, _ = ppo_model.predict(obs, deterministic=True)

    # Extract the actual closing price
    actual_closing_price = final_df['close'].iloc[-1]

    # Retrain PPO model at specified intervals
    if (datetime.now() - start_time).seconds >= retrain_interval_minutes * 60:
        print("Retraining PPO model with online data...")
        train_env = TradingEnvironment(final_df, config)  # Create a new training environment
        ppo_model.set_env(train_env)  # Update the PPO model's environment
        ppo_model.learn(total_timesteps=online_learning_steps)  # Retrain the model
        ppo_model.save(model_save_path)  # Save the updated model
        start_time = datetime.now()

    # Identify most recent high and low points
    recent_highs, recent_lows = find_local_extrema(final_df)

    most_recent_high = recent_highs[-1] if len(recent_highs) > 1 else None
    most_recent_low = recent_lows[-1] if len(recent_lows) > 1 else None

    high_trendline = [np.nan] * len(final_df)
    low_trendline = [np.nan] * len(final_df)

    if most_recent_high is not None:
        previous_high = recent_highs[-2] if len(recent_highs) > 2 else most_recent_high
        high_trendline = get_trendline(final_df, previous_high, most_recent_high, kind='high')

    if most_recent_low is not None:
        previous_low = recent_lows[-2] if len(recent_lows) > 2 else most_recent_low
        low_trendline = get_trendline(final_df, previous_low, most_recent_low, kind='low')

    # Prepare candlestick data for mplfinance
    actual_candles = final_df[-num_candles:].copy()

    # Create a DataFrame for mplfinance
    mpf_df = actual_candles[['open', 'high', 'low', 'close']]

    # Create addplot elements for predicted prices and actual close prices
    ap = [
        mpf.make_addplot(final_df['close'][-num_candles:], color='none', panel=0, secondary_y=False, label=f"Actual Price: {final_df['close'].iloc[-1]}"),
        #mpf.make_addplot(y_pred_ensemble_final_plot, color=(0.95, 0.38, 0.25, 1), panel=0, secondary_y=False, label=f'Predicted Prices ({y_pred_ensemble_final_plot[-1]:.2f})')
    ]

    # Add trendlines to the plot
    if most_recent_high is not None:
        ap.append(mpf.make_addplot(high_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    if most_recent_low is not None:
        ap.append(mpf.make_addplot(low_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    fig, axlist = mpf.plot(mpf_df, type='candle', style='binancedark', volume=False, addplot=ap,
                        title=f'Chart', ylabel='Price',
                        figsize=(14, 7), returnfig=True)

    for ax in axlist:
        ax.grid(False)

    axlist[0].legend()
    plt.show()

    # Execute market entry/exit logic
    market_entry_exit_logic(action, actual_closing_price, final_df)

    sleep_time = get_sleep_time(interval_minutes)
    print(f"Sleeping for {sleep_time}")
    time.sleep(sleep_time)